# Integrating Synthetic Data into a full ML Ops end-to-end system using SageMaker Pipelines service 



In [2]:
import boto3
import sagemaker


region = boto3.Session().region_name
role = sagemaker.get_execution_role()
default_bucket = sagemaker.session.Session().default_bucket()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


In [3]:
import yaml

with open("configs/config_stroke.yaml", 'r') as config_file:
    config = yaml.safe_load(config_file)

print(yaml.dump(config))

ML:
  ml_eval_metric: f1
  ml_metric_threshold: 0.0
  ml_task: classification
  objective: binary:logistic
  objective_type: Maximize
dataset:
  drop_columns: id
  name: healthcare-stroke-data
  path: s3://gretel-datasets/healthcare-dataset-stroke-data.csv
  target_column: stroke
gretel:
  generate_factor: 1.0
  strategy: balance
  target_balance: 1.0



### Get the pipeline instance

Here we get the pipeline instance from your pipeline module so that we can work with it.

In [4]:
from pipelines.scripts.pipeline import get_pipeline
from pipelines.scripts.datasets import datasets

# Change these to reflect your project/business name or if you want to separate ModelPackageGroup/Pipeline from the rest of your team
model_package_group_name = f"GretelModelPackageGroup-{config['dataset']['name']}"
pipeline_name = f"GretelPipeline-{config['dataset']['name']}"

pipeline = get_pipeline(
    region=region,
    role=role,
    default_bucket=default_bucket,
    model_package_group_name=model_package_group_name,
    pipeline_name=pipeline_name,
    config=config,
    use_gretel=False
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


/opt/conda/lib/python3.10/site-packages/sagemaker/workflow/pipeline_context.py:297: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


### Submit the pipeline to SageMaker and start execution

Let's submit our pipeline definition to the workflow service. The role passed in will be used by the workflow service to create all the jobs defined in the steps.

In [5]:
pipeline.upsert(role_arn=role)

INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/3e55b7a94c2fdecdfa25cc410f91da3e/sourcedir.tar.gz


Using provided s3_resource


INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/d740a32944cbfc42a3149407782d0920/runproc.sh
INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/a8a21575d9bca5dee603cf4cb3565503/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/b724fc16e3228521d3bb13c708323145/runproc.sh


Using provided s3_resource


Using provided s3_resource


INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/3e55b7a94c2fdecdfa25cc410f91da3e/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/d740a32944cbfc42a3149407782d0920/runproc.sh
INFO:sagemaker.processing:Uploaded /root/gretel-sagemaker-pipeline/pipelines/scripts to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/a8a21575d9bca5dee603cf4cb3565503/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-524473328983/GretelPipeline-healthcare-stroke-data/code/b724fc16e3228521d3bb13c708323145/runproc.sh


Using provided s3_resource


{'PipelineArn': 'arn:aws:sagemaker:us-east-1:524473328983:pipeline/GretelPipeline-healthcare-stroke-data',
 'ResponseMetadata': {'RequestId': 'b80d8056-8c57-49e1-ba5e-e3209379c32c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'b80d8056-8c57-49e1-ba5e-e3209379c32c',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '105',
   'date': 'Fri, 20 Oct 2023 20:55:58 GMT'},
  'RetryAttempts': 0}}

We'll start the pipeline, accepting all the default parameters.

Values can also be passed into these pipeline parameters on starting of the pipeline, and will be covered later. 

In [6]:
execution = pipeline.start()

### Pipeline Operations: examining and waiting for pipeline execution

Now we describe execution instance and list the steps in the execution to find out more about the execution.

In [7]:
# execution.describe()

We can wait for the execution by invoking `wait()` on the execution:

In [8]:
# execution.wait()

We can list the execution steps to check out the status and artifacts:

In [9]:
# execution.list_steps()

In [10]:
# # clean-up 

# client = boto3.client('sagemaker')
# pl_list = client.list_pipelines()
# for pl in pl_list['PipelineSummaries']:
#     print(f"Removing {pl['PipelineName']}")
#     client.delete_pipeline(PipelineName=pl['PipelineName'])